# 03. 3D モデル生成 (画像 → 3D / TripoSR)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hiirocreate/personalizeAI/blob/main/notebooks/03_3d_model.ipynb)

1 枚の画像から数秒でテクスチャ付き 3D モデル (GLB/OBJ) を生成します。
キャラクターは `01` で **正面・全身・Aポーズ・白背景** の画像を作ってから入力すると精度が上がります。

生成した GLB は [Web UI](https://hiirocreate.github.io/personalizeAI/#3d) でプレビュー可能。VTuber アバター(VRM)化の手順は README 参照。

In [ ]:
#@title ① 設定
USE_DRIVE = True  #@param {type:"boolean"}
LAUNCH_WEB_UI = False  #@param {type:"boolean"}
MC_RESOLUTION = 256  #@param [128, 256, 320] {type:"raw"}
TEXTURE = True  #@param {type:"boolean"}
FORMAT = "glb"  #@param ["glb", "obj"]
HF_TOKEN = ""

In [ ]:
#@title ② インストール (初回 5 分前後)

import os, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(cmd)
    return r.stdout

def dl(url, dst_dir, name=None):
    """aria2c で高速ダウンロード (既にあればスキップ)"""
    os.makedirs(dst_dir, exist_ok=True)
    name = name or url.split("/")[-1].split("?")[0]
    if os.path.exists(os.path.join(dst_dir, name)): return
    hdr = f'--header="Authorization: Bearer {HF_TOKEN}"' if HF_TOKEN and "huggingface.co" in url else ""
    print("↓", name); sh(f'aria2c -q -x16 -s16 -k1M --console-log-level=error {hdr} -d "{dst_dir}" -o "{name}" "{url}"')

if not os.path.exists("/usr/bin/aria2c"): sh("apt-get -qq install -y aria2")
if not os.path.exists("/content/personalizeAI"): sh("git clone -q --depth 1 https://github.com/hiirocreate/personalizeAI /content/personalizeAI")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/personalizeAI"
else:
    BASE = "/content/personalizeAI_data"
for d in ["loras", "outputs", "datasets", "3d"]: os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("データ保存先:", BASE)

T = "/content/TripoSR"
if not os.path.exists(T):
    sh(f"git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSR {T}")
    sh(f"cd {T} && pip install -q -r requirements.txt onnxruntime")
print("✅ 準備完了")

In [ ]:
#@title ③ 画像をアップロードして 3D 化 (LAUNCH_WEB_UI=True なら Gradio 共有URLを発行)
from google.colab import files
if LAUNCH_WEB_UI:
    subprocess.Popen(f"cd {T} && python gradio_app.py --share > /content/gradio.log 2>&1", shell=True)
    import time, re
    while True:
        m = re.search(r"https://\S+\.gradio\.live", open("/content/gradio.log").read() if os.path.exists("/content/gradio.log") else "")
        if m: print("🌐", m.group(0)); break
        time.sleep(3)
else:
    up = files.upload()
    for name in up:
        src = f"/content/{name}"
        out = f"{BASE}/3d/{os.path.splitext(name)[0]}"
        opts = f"--mc-resolution {MC_RESOLUTION} --model-save-format {FORMAT}" + (" --bake-texture --texture-resolution 1024" if TEXTURE else "")
        sh(f'cd {T} && python run.py "{src}" --output-dir "{out}" {opts}')
        print("✅", out, os.listdir(out + "/0"))

### さらに高品質にしたい場合 (無料 Web)
- [Hunyuan3D-2.1 (HF Space)](https://huggingface.co/spaces/tencent/Hunyuan3D-2.1) — 高精細メッシュ + PBR テクスチャ
- [TRELLIS (HF Space)](https://huggingface.co/spaces/trellis-community/TRELLIS) — 形状の破綻が少ない
- [Meshy](https://www.meshy.ai) / [Tripo](https://www.tripo3d.ai) — 無料枠あり、自動リギングも可